In [ ]:
%pip install -q requests==2.34.2 beautifulsoup4==4.15.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import hashlib
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("health_rag_data")
CACHE_DIR = DATA_DIR / "raw"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Working folder:", DATA_DIR.resolve())

Working folder: /content/health_rag_data


In [ ]:
BASE_URL = "https://www.niddk.nih.gov/health-information/digestive-diseases"
DISEASES = [
    "acid-reflux-ger-gerd-adults",
    "acid-reflux-ger-gerd-children",
    "acid-reflux-ger-gerd-infants",
    "appendicitis",
    "barretts-esophagus",
    "bowel-control-problems-fecal-incontinence",
    "celiac-disease",
    "chronic-diarrhea-children",
    "colon-polyps",
    "constipation",
    "constipation-children",
    "crohns-disease",
    "cyclic-vomiting-syndrome",
    "diarrhea",
    "diverticulosis-diverticulitis",
    "dumping-syndrome",
    "exocrine-pancreatic-insufficiency",
    "food-poisoning",
    "gallstones",
    "gas-digestive-tract",
    "gastritis-gastropathy",
    "gastrointestinal-bleeding",
    "gastroparesis",
    "hemorrhoids",
    "hirschsprung-disease",
    "indigestion-dyspepsia",
    "inguinal-hernia",
    "intestinal-pseudo-obstruction",
    "irritable-bowel-syndrome",
    "irritable-bowel-syndrome-children",
    "lactose-intolerance",
    "microscopic-colitis",
    "ostomy-surgery-bowel",
    "pancreatitis",
    "peptic-ulcers-stomach-ulcers",
    "proctitis",
    "short-bowel-syndrome",
    "ulcerative-colitis",
    "viral-gastroenteritis"
]

PAGES = {
    "facts": "definition-facts",
    "symptoms": "symptoms-causes",
    "diagnosis": "diagnosis",
    "treatment": "treatment",
    "nutrition": "eating-diet-nutrition",
    "clinical_trials": "clinical-trials",
}

SOURCES = [
    {
        "document_id": f"{disease}_{key}",
        "url": f"{BASE_URL}/{disease}/{path}",
    }
    for disease in DISEASES
    for key, path in PAGES.items()
]

for source in SOURCES:
    print(source["document_id"], "->", source["url"])

acid-reflux-ger-gerd-adults_facts -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/definition-facts
acid-reflux-ger-gerd-adults_symptoms -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/symptoms-causes
acid-reflux-ger-gerd-adults_diagnosis -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/diagnosis
acid-reflux-ger-gerd-adults_treatment -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/treatment
acid-reflux-ger-gerd-adults_nutrition -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/eating-diet-nutrition
acid-reflux-ger-gerd-adults_clinical_trials -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-reflux-ger-gerd-adults/clinical-trials
acid-reflux-ger-gerd-children_facts -> https://www.niddk.nih.gov/health-information/digestive-diseases/acid-re

In [ ]:
def fetch_page(source):
    cache_path = CACHE_DIR / f"{source['document_id']}.json"

    if cache_path.exists():
        cached = json.loads(cache_path.read_text(encoding="utf-8"))
        if cached["requested_url"] != source["url"]:
            raise ValueError("Cached URL changed. Inspect or remove: " + str(cache_path))
        return cached

    response = requests.get(
        source["url"],
        headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"},
        timeout=(10, 30),
    )

    response.raise_for_status()
    if "html" not in response.headers.get("Content-Type", "").lower():
        raise ValueError("Expected an HTML article: " + source["url"])
    if urlparse(response.url).hostname not in {"www.niddk.nih.gov", "niddk.nih.gov"}:
        raise ValueError("Source moved to a different host. Review: " + response.url)

    cached = {
        "requested_url": source["url"],
        "source_url": response.url,
        "fetched_at": datetime.now(timezone.utc).isoformat(),
        "html": response.content.decode("utf-8"),
    }
    cache_path.write_text(json.dumps(cached, ensure_ascii=False, indent=2), encoding="utf-8")
    time.sleep(1)  # A small pause between new downloads.
    return cached


In [ ]:
sample_raw = fetch_page(SOURCES[0])
sample_soup = BeautifulSoup(sample_raw["html"], "html.parser")
sample_article = sample_soup.select_one("article.dk-content")

if sample_article is None:
    raise ValueError("Article selector no longer matches. Inspect the source HTML before continuing.")

print("Title:", sample_article.find("h1").get_text(" ", strip=True))
review = sample_article.select_one(".dk-review-date")
print("Publisher review:", review.get_text(" ", strip=True) if review else "Not provided")
print("Downloaded:", sample_raw["fetched_at"])
print("Article body found:", sample_article.select_one(".health-detail-content") is not None)


Title: Definition & Facts for GER & GERD
Publisher review: Last Reviewed July 2020
Downloaded: 2026-09-14T08:29:17.240740+00:00
Article body found: True


In [ ]:
def extract_document(source, raw):
    soup = BeautifulSoup(raw["html"], "html.parser")
    article = soup.select_one("article.dk-content")
    if article is None:
        raise ValueError("Article selector missing: " + raw["source_url"])

    title_node = article.find("h1")
    body = article.select_one(".health-detail-content")
    if title_node is None or body is None:
        raise ValueError("Title or article body missing: " + raw["source_url"])

    title = title_node.get_text(" ", strip=True)
    review_node = article.select_one(".dk-review-date")
    last_reviewed = review_node.get_text(" ", strip=True) if review_node else None

    for element in body.select("script, style, nav, figure"):
        element.decompose()
    for paragraph in list(body.find_all("p")):
        if paragraph.get_text(" ", strip=True).lower() == "in this section:":
            contents_list = paragraph.find_next_sibling()
            if contents_list and contents_list.name in {"ul", "ol"}:
                contents_list.decompose()
            paragraph.decompose()

    sections = []
    heading = title
    parent_heading = ""
    anchor = ""
    paragraphs = []
    references = []
    in_references = False

    def save_section():
        if paragraphs:
            sections.append({
                "heading": heading,
                "source_url": raw["source_url"] + ("#" + anchor if anchor else ""),
                "text": "\n\n".join(paragraphs),
            })

    for element in body.find_all(["h2", "h3", "p", "li", "tr"]):
        # A list item or table row already includes its nested text.
        if element.find_parent(["li", "table"]) and element.name != "tr":
            continue
        if element.name == "tr":
            text = " | ".join(cell.get_text(" ", strip=True) for cell in element.find_all(["th", "td"]))
        else:
            text = element.get_text(" ", strip=True)
        text = re.sub(r"\s+([,.;:!?])", r"\1", text)
        if not text:
            continue

        if element.name in {"h2", "h3"}:
            save_section()
            paragraphs = []
            in_references = text.lower() in {"references", "reference"}
            if element.name == "h2":
                parent_heading = text
                heading = text
            else:
                heading = f"{parent_heading} / {text}" if parent_heading else text
            anchor = element.get("id", "")
        elif in_references:
            references.append(text)
        else:
            paragraphs.append(("- " if element.name == "li" else "") + text)
    save_section()

    text = "\n\n".join(section["heading"] + "\n" + section["text"] for section in sections)
    if len(text.split()) < 30:
        raise ValueError("Unexpectedly little article text; inspect: " + raw["source_url"])

    return {
        "document_id": source["document_id"],
        "title": title,
        "publisher": "National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK)",
        "source_url": raw["source_url"],
        "fetched_at": raw["fetched_at"],
        "last_reviewed": last_reviewed,
        "reuse_policy_url": "https://www.niddk.nih.gov/copyright",
        "content_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
        "sections": sections,
        "references": references,
        "text": text,
    }


In [ ]:
sample_document = extract_document(SOURCES[0], sample_raw)

print("Document:", sample_document["title"])
print("Sections:", len(sample_document["sections"]))
print("Words:", len(sample_document["text"].split()))
print("\nSection headings:")
for section in sample_document["sections"]:
    print("-", section["heading"])

print("\nFirst section preview:\n")
print(sample_document["sections"][0]["text"][:600])


Document: Definition & Facts for GER & GERD
Sections: 11
Words: 392

Section headings:
- What is GER?
- Does GER have another name?
- How common is GER?
- What is GERD?
- How common is GERD?
- Who is more likely to have GERD?
- What are the complications of GERD?
- What are the complications of GERD? / Esophagitis
- What are the complications of GERD? / Esophageal stricture
- What are the complications of GERD? / Barrett’s esophagus
- What are the complications of GERD? / Complications outside the esophagus

First section preview:

Gastroesophageal reflux (GER) happens when your stomach contents come back up into your esophagus. Many people have GER once in a while, and GER often happens without causing symptoms. In some cases, GER may cause heartburn, also called acid indigestion.


In [ ]:
documents = []

documents = []
skipped_sources = []

for source in SOURCES:
    try:
        raw = fetch_page(source)
        document = extract_document(source, raw)

    except requests.exceptions.HTTPError as error:
        if error.response.status_code == 404:
            skipped_sources.append({
                "document_id": source["document_id"],
                "url": source["url"],
                "reason": "Page not available",
            })

            print(f"SKIP | {source['document_id']} | page not available")
            continue

        # Other HTTP errors should still stop the process.
        raise RuntimeError(
            f"Could not collect {source['url']}: {error}"
        ) from error

    except (requests.RequestException, ValueError, KeyError) as error:
        raise RuntimeError(
            f"Could not collect {source['url']}: {error}"
        ) from error

    documents.append(document)

    print(
        f"OK | {document['document_id']} | "
        f"{len(document['sections'])} sections | "
        f"{len(document['text'].split())} words"
    )

OK | acid-reflux-ger-gerd-adults_facts | 11 sections | 392 words
OK | acid-reflux-ger-gerd-adults_symptoms | 2 sections | 384 words
OK | acid-reflux-ger-gerd-adults_diagnosis | 4 sections | 406 words
OK | acid-reflux-ger-gerd-adults_treatment | 4 sections | 661 words
OK | acid-reflux-ger-gerd-adults_nutrition | 2 sections | 180 words
OK | acid-reflux-ger-gerd-adults_clinical_trials | 3 sections | 236 words
OK | acid-reflux-ger-gerd-children_facts | 11 sections | 529 words
OK | acid-reflux-ger-gerd-children_symptoms | 2 sections | 558 words
OK | acid-reflux-ger-gerd-children_diagnosis | 5 sections | 468 words
OK | acid-reflux-ger-gerd-children_treatment | 4 sections | 547 words
OK | acid-reflux-ger-gerd-children_nutrition | 2 sections | 178 words
OK | acid-reflux-ger-gerd-children_clinical_trials | 5 sections | 532 words
OK | acid-reflux-ger-gerd-infants_facts | 10 sections | 487 words
OK | acid-reflux-ger-gerd-infants_symptoms | 2 sections | 519 words
OK | acid-reflux-ger-gerd-infants_

In [ ]:
output_path = DATA_DIR / "health_documents.json"
temporary_path = output_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(documents, ensure_ascii=False, indent=2), encoding="utf-8")
temporary_path.replace(output_path)

loaded_documents = json.loads(output_path.read_text(encoding="utf-8"))
assert loaded_documents == documents
print(f"Saved {len(loaded_documents)} documents to {output_path.resolve()}")
print("Bytes:", output_path.stat().st_size)


Saved 221 documents to /content/health_rag_data/health_documents.json
Bytes: 1628685
